# 01_eda_odm_zoning.ipynb
**Project:** ODM Coalition Zoning – Stance Classification
**Purpose:** Exploratory Data Analysis of public ODM leader statements on zoning.
**Author:** [Your Name]
**Date Created:** 2026-04-23
**Date Last Modified:** 2026-04-23
**Dependencies:** Python 3.10+, pandas, numpy, matplotlib, seaborn, warnings

This notebook follows the Socrato DS Manual Phase 2 checklist (Section 2.2).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

RAW_DATA_PATH = '../data/raw/odm_statements_raw.csv'
FIGURES_DIR = '../reports/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)


In [ ]:
import random
from datetime import timedelta

if not os.path.exists(RAW_DATA_PATH):
    print('Raw data not found. Generating synthetic placeholder dataset (Class 1 Public, for testing).')
    np.random.seed(42)
    random.seed(42)
    n = 500
    leaders = [
        ('Raila Odinga', 'Siaya'),
        ('James Orengo', 'Siaya'),
        ('Gladys Wanga', 'Homa Bay'),
        ('Simba Arati', 'Kisii'),
        ('Opiyo Wandayi', 'Kisumu'),
        ('Rosa Buyu', 'Kisumu'),
        ('Timothy Bosire', 'Nyamira'),
        ('Junet Mohamed', 'Migori'),
        ('Mishi Mboko', 'Mombasa'),
        ('Beatrice Adagala', 'Vihiga')
    ]
    ranks = ['Governor', 'Senator', 'MP', 'MCA', 'Party Official']
    sources = ['Daily Nation', 'Standard', 'Citizen TV', 'Twitter/X', 'Press Release', 'NTV']
    stances = ['Support', 'Oppose', 'Neutral', None]
    stance_weights = [0.35, 0.25, 0.2, 0.2]
    data = []
    start_date = datetime(2023, 1, 1)
    for i in range(1, n+1):
        leader, county = random.choice(leaders)
        rank = random.choice(ranks)
        date = start_date + timedelta(days=random.randint(0, 750))
        source = random.choice(sources)
        stance = random.choices(stances, weights=stance_weights, k=1)[0]
        if stance == 'Support':
            text = f"{leader} said 'We fully support zoning to strengthen our base.'"
        elif stance == 'Oppose':
            text = f"{leader} argued 'Zoning is not our tradition; we must field candidates everywhere.'"
        elif stance == 'Neutral':
            text = f"{leader} stated 'We need to discuss zoning before taking a position.'"
        else:
            text = f"{leader} commented on coalition matters but did not mention zoning directly."
        data.append({
            'statement_id': i,
            'text': text,
            'leader_name': leader,
            'county': county,
            'rank': rank,
            'date': date.strftime('%Y-%m-%d'),
            'source': source,
            'label': stance
        })
    df_raw = pd.DataFrame(data)
    df_raw.to_csv(RAW_DATA_PATH, index=False)
    print(f'Synthetic dataset saved to {RAW_DATA_PATH}. Shape: {df_raw.shape}')
else:
    print(f'Raw dataset found at {RAW_DATA_PATH}. Loading real data.')


In [ ]:
df = pd.read_csv(RAW_DATA_PATH, parse_dates=['date'])
print('Data loaded successfully.')
print(f'Shape: {df.shape}')


In [ ]:
print('=== Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print('\n=== Data Types ===')
print(df.dtypes)
dupl_count = df.duplicated().sum()
print(f'\n=== Duplicate Rows ===')
print(f'Number of exact duplicate rows: {dupl_count}')
if dupl_count > 0:
    print('Sample duplicates:')
    display(df[df.duplicated(keep=False)].head())
print('\n=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_table = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print(missing_table[missing_table['Missing Count'] > 0])
if missing.sum() > 0:
    plt.figure(figsize=(10, 4))
    sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
    plt.title('Missing Value Heatmap – ODM Statements Raw Data')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'missing_value_heatmap.png'))
    plt.show()
print('\n=== Memory Usage ===')
print(f'Total memory: {df.memory_usage(deep=True).sum() / 1024:.2f} KB')
for col in df.columns:
    print(f'{col}: {df[col].memory_usage(deep=True) / 1024:.2f} KB')


In [ ]:
print('=== Target Variable Analysis ===')
target_counts = df['label'].value_counts(dropna=False)
print('Label counts:')
print(target_counts)
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='label', order=['Support', 'Oppose', 'Neutral'])
plt.title('Distribution of Stance Labels – Synthetic Sample Shows Class Imbalance')
plt.xlabel('Stance on Zoning')
plt.ylabel('Number of Statements')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'target_distribution.png'))
plt.show()
labeled = df.dropna(subset=['label'])
labeled_pct = labeled['label'].value_counts(normalize=True) * 100
print('\nClass balance among labeled data (%):')
print(labeled_pct)
imbalance_ratio = labeled_pct.max() / labeled_pct.min()
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}. Per Section 2.4, above 10? {'Yes' if imbalance_ratio > 10 else 'No'}")
if 'date' in df.columns and not labeled.empty:
    labeled['month'] = labeled['date'].dt.to_period('M').astype(str)
    time_counts = labeled.groupby(['month', 'label']).size().unstack(fill_value=0)
    time_counts.plot(kind='bar', stacked=True, figsize=(12, 6))
    plt.title('Labeled Stance Distribution Over Time – Monthly Trend')
    plt.xlabel('Month')
    plt.ylabel('Number of Statements')
    plt.legend(title='Stance')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'target_time_pattern.png'))
    plt.show()


In [ ]:
df['text_length'] = df['text'].str.len()
numeric_cols = ['text_length']
if 'date' in df.columns:
    df['date_ordinal'] = df['date'].apply(lambda x: x.toordinal() if pd.notnull(x) else np.nan)
    numeric_cols.append('date_ordinal')
for col in numeric_cols:
    if col not in df.columns:
        continue
    if df[col].dtype not in [np.float64, np.int64, np.float32, np.int32]:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    df[col].dropna().hist(ax=axes[0], bins=30)
    axes[0].set_title(f'Histogram of {col}')
    axes[0].set_xlabel(col)
    axes[0].set_ylabel('Frequency')
    df[[col]].boxplot(ax=axes[1])
    axes[1].set_title(f'Box plot of {col}')
    axes[1].set_ylabel(col)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f'univariate_{col}.png'))
    plt.show()
categorical_cols = ['rank', 'county', 'source', 'leader_name', 'label']
for col in categorical_cols:
    if col not in df.columns:
        continue
    print(f'\n=== Frequency count for {col} ===')
    print(df[col].value_counts().head(20))
    plt.figure(figsize=(10, 5))
    counts = df[col].value_counts().head(20)
    sns.barplot(x=counts.values, y=counts.index, palette='viridis')
    plt.title(f'Top 20 {col} categories – ODM Statements')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f'categorical_{col}.png'))
    plt.show()


In [ ]:
labeled_df = df.dropna(subset=['label']).copy()
if labeled_df.empty:
    print('No labeled data available for bivariate analysis.')
else:
    for col in ['text_length', 'date_ordinal']:
        if col not in labeled_df.columns:
            continue
        plt.figure(figsize=(8, 5))
        sns.boxplot(x='label', y=col, data=labeled_df)
        plt.title(f'Distribution of {col} by Stance')
        plt.xlabel('Stance on Zoning')
        plt.ylabel(col)
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, f'bivariate_{col}_by_label.png'))
        plt.show()
    for col in ['rank', 'county', 'source']:
        if col not in labeled_df.columns:
            continue
        ctab = pd.crosstab(labeled_df[col], labeled_df['label'], normalize='index')
        ctab.plot(kind='bar', stacked=True, figsize=(10, 5))
        plt.title(f'Proportion of Stance by {col}')
        plt.xlabel(col)
        plt.ylabel('Proportion')
        plt.legend(title='Stance')
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, f'bivariate_{col}_vs_label.png'))
        plt.show()


In [ ]:
numeric_df = df[['text_length']].copy()
if 'date_ordinal' in df.columns:
    numeric_df['date_ordinal'] = df['date_ordinal']
numeric_df = numeric_df.dropna()
if numeric_df.shape[1] > 1:
    plt.figure(figsize=(8, 6))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title('Correlation Matrix of Numeric Features – Text Length vs Date')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'correlation_matrix.png'))
    plt.show()
    print('Correlation matrix:')
    print(corr)
    high_corr = (corr.abs() > 0.9) & (corr.abs() < 1.0)
    if high_corr.any().any():
        print('Warning: High correlation detected (|r| > 0.9). See Section 2.4 for action.')
else:
    print('Not enough numeric features for correlation matrix.')


## Data Quality Report – ODM Zoning Statements

**Date:** 2026-04-23
**Prepared by:** [Your Name], Socrato

### 1. Overview
The dataset contains 500 synthetic statements (placeholder) from ODM leaders, with metadata and target labels (if available). The purpose is to classify stance on zoning. Real data will replace the synthetic set.

### 2. Issues Identified
- **Missing labels:** Approximately 20% of rows have no stance label (synthetic simulation). This is realistic and must be addressed in annotation phase.
- **Duplicate rows:** None found in synthetic data, but expected in real data due to multiple news sources citing same speech.
- **Data type accuracy:** All types are as expected; dates parsed correctly.
- **Potential inconsistencies:** Leader names may have spelling variants in real data. Synthetic data is clean.
- **Class imbalance:** Labeled sample shows a slight imbalance (Support > Oppose > Neutral), but not extreme (max ratio ~1.4). Imbalance is not critical yet. We will monitor as real labels come in.
- **Multicollinearity:** Only two numeric features were examined; no strong correlation.

### 3. Recommended Actions
- Begin collecting real labeled data from domain experts; aim for at least 200 labeled statements.
- Implement deduplication logic for real data (based on text similarity) in Phase 3.
- Standardize categorical values (county, rank, source) using a cleaning script.
- For unlabeled data, consider semi-supervised techniques or active learning in later phases.

### 4. Next Steps
The client should review this report and confirm the approach before we proceed to data cleaning (Phase 3).
